# **Preprocesamiento de datos**

In [109]:
import pandas as pd
import numpy as np
import re

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter, MaxNLocator

In [110]:
dataset = pd.read_csv("data/raw.csv")

# Nuevas columnas

### Turbo

Buscar "T", "turbo" en la columna "Motor". También podríamos buscar en la descripción, versión o el título (ahora es solo en "Motor")

In [111]:
def is_turbo(row):
    # Combine Motor and Versión columns as strings (handle NaN)
    motor = str(row.get('Motor', ''))
    version = str(row.get('Versión', ''))
    combined = f"{motor} {version}"
    # Look for 'turbo' or typical turbo patterns
    if re.search(r'turbo', combined, re.IGNORECASE):
        return 1
    if re.search(r'(?:\d[\.,]?\d?\s*[tT]|TSI|TFSI|TURBO)', combined, re.IGNORECASE):
        return 1
    return 0

# Apply row-wise
dataset['Turbo'] = dataset.apply(is_turbo, axis=1)

### Caballos de fuerza (cv)


In [112]:
def extract_cv(text):
    if not isinstance(text, str):
        return np.nan
    match = re.search(r'(\d+)\s*[cC][vV]\b', text)
    if match:
        return int(match.group(1))
    return np.nan

# Solo extrae el valor de CV, no modifica la columna 'Versión'
dataset['cv'] = dataset['Versión'].apply(extract_cv)

# Transformación de columnas

## **Motor**

### Buscar el tamaño del motor en "Versión" si no está en motor

Ejemplo: Fila 86 del archivo @data/raw.csv 
- `Motor` es **TURBOALIMENTADO** y `Versión` es **2.0 T ALLSPACE COMFORTLINE AT L17**

In [113]:
def extract_engine_size(version):
    # Look for patterns like "2.0", "1.6", etc. at the start of the version string
    if not isinstance(version, str):
        return None
    match = re.match(r'(\d\.\d)', version)
    if match:
        return match.group(1)
    return None

def fix_motor(row):
    motor = row['Motor']
    version = row['Versión']
    try:
        float(motor)
        is_number = True
    except (ValueError, TypeError):
        is_number = False
    if not is_number:
        engine_size = extract_engine_size(version)
        if engine_size:
            return engine_size
    return motor

dataset['Motor'] = dataset.apply(fix_motor, axis=1)

print(dataset.iloc[84])

Unnamed: 0                                                                84
Marca                                                             Volkswagen
Modelo                                                                Tiguan
Año                                                                   2018.0
Versión                                    2.0 T ALLSPACE COMFORTLINE AT L17
Color                                                                 BLANCA
Tipo de combustible                                                    Nafta
Puertas                                                                  5.0
Transmisión                                                       Automática
Motor                                                                    2.0
Tipo de carrocería                                                       SUV
Con cámara de retroceso                                                  NaN
Kilómetros                                                           89000.0

## **Versión**

### Borrar el cilindraje.

Ejemplo: borrar 2.4 de "2.4 Sport At"

In [ ]:
def remove_cilinder_size(row):
    motor = str(row['Motor']) if pd.notnull(row['Motor']) else ''
    version = str(row['Versión']) if pd.notnull(row['Versión']) else ''
    if motor and version.startswith(motor + ' '):
        return version[len(motor) + 1:]
    return version

dataset['Versión'] = dataset.apply(remove_cilinder_size, axis=1)

### Borrar "at" y "mt" de la columna
(Manual/Automatic transmission) 

Ya existe una columna para eso, pero algunos datos tienen la cantidad de marchas que tiene la transmisión (6at o AT8)

In [115]:
def remove_transmission(text):
    if not isinstance(text, str):
        return text
    return re.sub(r'\b(at\d*|at|mt|dct|cvt)\b', '', text, flags=re.IGNORECASE).replace('  ', ' ').strip()

dataset['Versión'] = dataset['Versión'].apply(remove_transmission)

# Resultado final

In [116]:
print(dataset['cv'].isna().sum(), "CVs sin datos")

dataset.to_csv("data/cleaned.csv", index=False)

13295 CVs sin datos
